# Mol3D Experiment — Results Analysis (HP-Tuned)

Compares CT (coords / simple / full), GNN (GCN / GAT / GIN), CIN (CIN / CIN++), and SchNet on the Mol3D dataset. CT/GNN/CIN use each model's tuned hyperparameters from `hp_tuning_<model>.py` (`results/best_hp_<model>.json`) instead of hardcoded CLI defaults. SchNet does **not** have a hp-tuned run (`best_hp_schnet.json` doesn't exist, tuning was never attempted), so it uses its plain (non-tuned) result instead: `results_mol3d_schnet.json`. See `analyze_results.ipynb` for the fully non-tuned baseline comparison.

Task: molecular property regression (single target).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from IPython.display import display

plt.rcParams['figure.dpi'] = 130
pd.set_option('display.float_format', '{:.4f}'.format)

RESULTS_DIR = Path('results')

In [ ]:
FILES = {
    'CT-coords': 'results_mol3d_ct_coords_hptuned.json',
    'CT-simple': 'results_mol3d_ct_simple_hptuned.json',
    'CT-full':   'results_mol3d_ct_full_hptuned.json',
    'GCN':       'results_mol3d_gcn_hptuned.json',
    'GAT':       'results_mol3d_gat_hptuned.json',
    'GIN':       'results_mol3d_gin_hptuned.json',
    'CIN':       'results_mol3d_cin_hptuned.json',
    'CINpp':     'results_mol3d_cinpp_hptuned.json',
    'SchNet':    'results_mol3d_schnet.json',
}
# SchNet is NOT hp-tuned (no best_hp_schnet.json, tuning never run) -- this is the
# plain result, included as the best available SchNet numbers.

data = {}
for label, fname in FILES.items():
    p = RESULTS_DIR / fname
    if p.exists():
        with open(p) as f:
            data[label] = json.load(f)
    else:
        print(f'Missing: {fname}')

print(f'Loaded {len(data)} result files: {list(data.keys())}')

## Summary Table

In [ ]:
rows = []
for label, d in data.items():
    run_rmses = [r['test_rmse'] for r in d['runs']]
    run_maes  = [r['test_mae']  for r in d['runs']]
    run_r2s   = [r['test_r2']   for r in d['runs']]
    rows.append({
        'model':      label,
        'num_params': d.get('num_params'),
        'RMSE':       d['mean_test_rmse'],
        'RMSE_std':   round(float(np.std(run_rmses)), 4),
        'MAE':        d['mean_test_mae'],
        'MAE_std':    round(float(np.std(run_maes)),  4),
        'R²':         d['mean_test_r2'],
        'R²_std':     round(float(np.std(run_r2s)),   4),
    })

df = pd.DataFrame(rows).set_index('model')
df.sort_values('RMSE', inplace=True)
display(df.style
        .format(precision=4)
        .background_gradient(subset=['RMSE', 'MAE'], cmap='RdYlGn_r', axis=0)
        .background_gradient(subset=['R²'],          cmap='RdYlGn',   axis=0))

## Metric Comparison — Bar Charts

In [ ]:
COLORS = {
    'CT':     '#4C72B0',
    'GCN':    '#DD8452',
    'GAT':    '#55A868',
    'GIN':    '#C44E52',
    'CINpp':  '#937860',
    'CIN':    '#8172B2',
    'SchNet': '#64B5CD',
}

def model_color(label):
    for prefix, c in COLORS.items():
        if label.startswith(prefix):
            return c
    return 'gray'

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (metric, col, cmap_dir) in zip(axes, [
        ('RMSE', 'RMSE', 'lower'), ('MAE', 'MAE', 'lower'), ('R²', 'R²', 'higher')]):
    vals   = df[col].dropna()
    stds   = df.loc[vals.index, f'{col}_std']
    colors = [model_color(m) for m in vals.index]
    ax.bar(range(len(vals)), vals.values, color=colors,
           yerr=stds.values, capsize=4, error_kw={'linewidth': 1.2})
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, rotation=35, ha='right', fontsize=9)
    ax.set_title(f'{metric}  ({cmap_dir} is better)', fontsize=11)
    ax.set_ylabel(metric)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=k) for k, c in COLORS.items()]
fig.legend(handles=legend_elements, loc='upper right', fontsize=8,
           bbox_to_anchor=(1.04, 0.95))
plt.suptitle('Test metrics — mean ± std over 3 runs (SchNet non-tuned, rest hp-tuned)', fontsize=13)
plt.tight_layout()
plt.show()

## Training Loss Curves

In [ ]:
GROUPS = {
    'CT variants':  [k for k in data if k.startswith('CT')],
    'GNNs':         [k for k in data if k in ('GCN', 'GAT', 'GIN')],
    'CIN variants': [k for k in data if k in ('CIN', 'CINpp')],
    'SchNet':       [k for k in data if k == 'SchNet'],
}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (group_name, models) in zip(axes, GROUPS.items()):
    cmap = cm.get_cmap('tab10', max(len(models), 1))
    for idx, label in enumerate(models):
        if label not in data:
            continue
        d = data[label]
        all_losses = [r['train_losses'] for r in d['runs']]
        min_len = min(len(l) for l in all_losses)
        mean_losses = np.mean([l[:min_len] for l in all_losses], axis=0)
        color = cmap(idx)
        for l in all_losses:
            ax.plot(l[:min_len], color=color, alpha=0.25, linewidth=0.8)
        ax.plot(mean_losses, color=color, linewidth=2, label=label)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Train Loss')
    ax.set_title(group_name)
    ax.legend(fontsize=8)

plt.suptitle('Training loss (faint = individual runs, bold = mean) — SchNet non-tuned, rest hp-tuned', fontsize=12)
plt.tight_layout()
plt.show()

## Predicted vs. True (Run 1)

In [ ]:
n_models = len(data)
ncols = 4
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4))
axes = axes.flatten()

for ax, (label, d) in zip(axes, data.items()):
    preds = d['runs'][0].get('predictions', [])
    if not preds:
        ax.set_visible(False)
        continue
    true_vals = np.array([p['true'] for p in preds])
    pred_vals = np.array([p['pred'] for p in preds])
    vmin, vmax = true_vals.min(), true_vals.max()
    ax.scatter(true_vals, pred_vals, alpha=0.3, s=5, rasterized=True)
    ax.plot([vmin, vmax], [vmin, vmax], 'r--', linewidth=1)
    rmse = d['runs'][0]['test_rmse']
    r2   = d['runs'][0]['test_r2']
    ax.set_title(f'{label}\nRMSE={rmse:.3f}  R²={r2:.3f}', fontsize=9)
    ax.set_xlabel('True')
    ax.set_ylabel('Predicted')

for ax in axes[len(data):]:
    ax.set_visible(False)

plt.suptitle('Predicted vs. True values (run 1) — hp-tuned', fontsize=13)
plt.tight_layout()
plt.show()

## Error Distribution (Run 1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for label, d in data.items():
    preds = d['runs'][0].get('predictions', [])
    if not preds:
        continue
    errors = np.array([p['pred'] - p['true'] for p in preds])
    ax.hist(errors, bins=60, alpha=0.5, label=f'{label} (MAE={np.abs(errors).mean():.3f})',
            density=True, histtype='stepfilled')

ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Prediction error  (pred − true)')
ax.set_ylabel('Density')
ax.set_title('Error distributions (run 1) — hp-tuned')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Runtime & Parameter Count

In [ ]:
rt_rows = []
for label, d in data.items():
    runtimes = [r.get('runtime') for r in d['runs'] if r.get('runtime')]
    rt_rows.append({
        'model':          label,
        'num_params':     d.get('num_params'),
        'total_runtime_h': round(sum(runtimes) / 3600, 2) if runtimes else None,
        'per_run_h':       round(np.mean(runtimes) / 3600, 2) if runtimes else None,
    })

rt_df = pd.DataFrame(rt_rows).set_index('model').sort_values('per_run_h')
display(rt_df)

## MAE Table (for slides)

In [ ]:
mae_df = df[['MAE', 'MAE_std']].sort_values('MAE').reset_index()

fig, ax = plt.subplots(figsize=(4.5, 0.5 + 0.45 * len(mae_df)))
ax.axis('off')

table = ax.table(
    cellText=[[m, f'{mae:.4f} ± {std:.4f}'] for m, mae, std in mae_df.values],
    colLabels=['Model', 'MAE'],
    cellLoc='center',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 1.8)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('black')
    if row == 0:
        cell.set_text_props(weight='bold')

plt.tight_layout()
plt.savefig('mae_table_hptuned.jpg', dpi=300, bbox_inches='tight')
plt.show()

## Model Feature Overview (for slides)

In [ ]:
FEATURE_OVERVIEW = [
    ('CT-coords', 'xyz coordinates',
                  'bond midpoint (xyz)',
                  'ring centroid (xyz)'),
    ('CT-simple', 'mass, electronegativity,\nvdW radius',
                  'bond type',
                  'ring size'),
    ('CT-full',   'atomic #, valence, degree,\naromaticity, chirality, charge,\nhybridization, xyz\n+ positional encoding',
                  'bond type, conjugation,\nin-ring, stereo, ΔEN,\nmin ring size\n+ positional encoding',
                  'size, aromaticity,\nheteroatom count, fusion,\navg EN\n+ positional encoding'),
    ('GCN',       'same as CT-full (11-d)', '—', '—'),
    ('GAT',       'same as CT-full (11-d)', 'same as CT-full (8-d)', '—'),
    ('GIN',       'same as CT-full (11-d)', '—', '—'),
    ('CIN',       'same as CT-full\n(no xyz, no PE)', 'same as CT-full\n(no PE)', 'same as CT-full\n(no PE)'),
    ('CINpp',     'same as CT-full\n(no xyz, no PE)', 'same as CT-full\n(no PE)', 'same as CT-full\n(no PE)'),
    ('SchNet',    'atomic number only\n(+ RDKit-embedded 3D\nconformer coordinates)', '— (implicit, via learned\ndistance cutoff)', '—'),
]

fig, ax = plt.subplots(figsize=(13, 0.5 + 0.95 * len(FEATURE_OVERVIEW)))
ax.axis('off')

table = ax.table(
    cellText=[list(row[1:]) for row in FEATURE_OVERVIEW],
    rowLabels=[row[0] for row in FEATURE_OVERVIEW],
    colLabels=['Atom features', 'Edge features', 'Ring features'],
    cellLoc='center',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 3.2)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('black')
    cell.set_text_props(ha='center', va='center')
    if row == 0 or col == -1:
        cell.set_text_props(weight='bold', ha='center', va='center')

plt.tight_layout()
plt.savefig('feature_table_hptuned.jpg', dpi=300, bbox_inches='tight')
plt.show()

## CIN / CIN++ — Model-Specific Hasse Graph Diameter

The CT Hasse graph diameter (see `mol3d/hasse_graph_diameter.ipynb`, or the CT Hasse section in `mol3d_fullerene/analyze_results_hptuned.ipynb`) is built from CT's 6 neighborhoods (`adj00`/`icd01`/`adj11`/`icd02`/`icd12`/`adj22`). CIN and CIN++ message-pass over a *different* set of neighborhoods (`hasse_graph_diameter_cin.py`, mirroring `models/cin.py::CINLayer.forward`): CIN uses `up0`/`up1`/`boundary1`/`boundary2` only, CIN++ adds `down1`/`down2`, and neither ever has an atom↔ring relation the way CT's `icd02` does. So each variant gets its own Hasse graph diameter here, computed directly from the index tensors each model actually trains on (test split only).

Shown for every model (not just CIN/CINpp): even models that don't use rings at all are still evaluated on the same test molecules, so their error can still be checked against this molecule-level metric.

In [ ]:
cin_hasse_npz = np.load("hasse_graph_cin_per_molecule.npz")
cin_hasse_pos = {int(idx): i for i, idx in enumerate(cin_hasse_npz["index"])}
cin_hd_all = cin_hasse_npz["cin_hasse_diameter"]
cinpp_hd_all = cin_hasse_npz["cinpp_hasse_diameter"]

print(f"CIN   hasse diameter: n={len(cin_hd_all)}  min={cin_hd_all.min()} max={cin_hd_all.max()} mean={cin_hd_all.mean():.2f}")
print(f"CINpp hasse diameter: n={len(cinpp_hd_all)}  min={cinpp_hd_all.min()} max={cinpp_hd_all.max()} mean={cinpp_hd_all.mean():.2f}")
print(f"Correlation(CIN, CINpp hasse diameter): {np.corrcoef(cin_hd_all, cinpp_hd_all)[0,1]:.4f}")
n_differ = (cin_hd_all != cinpp_hd_all).sum()
print(f"Differ on {n_differ}/{len(cin_hd_all)} molecules ({100*n_differ/len(cin_hd_all):.1f}%)")

In [ ]:
def per_molecule_err(d):
    err_by_idx = {}
    for run in d["runs"]:
        for p in run["predictions"]:
            err_by_idx.setdefault(p["index"], []).append(abs(p["pred"] - p["true"]))
    return {k: float(np.mean(v)) for k, v in err_by_idx.items()}

model_labels_all = list(data.keys())
n_models_all = len(model_labels_all)
err_maps_all = {lbl: per_molecule_err(data[lbl]) for lbl in model_labels_all}

fig, axes = plt.subplots(2, n_models_all, figsize=(3.0 * n_models_all, 5.6), squeeze=False)

for row, (hd_arr, hd_name) in enumerate([(cin_hd_all, "CIN Hasse diameter"), (cinpp_hd_all, "CINpp Hasse diameter")]):
    for col, label in enumerate(model_labels_all):
        ax = axes[row, col]
        err_by_idx = err_maps_all[label]
        idxs = np.array(sorted(i for i in err_by_idx if i in cin_hasse_pos))
        x = hd_arr[[cin_hasse_pos[i] for i in idxs]].astype(float)
        y = np.array([err_by_idx[i] for i in idxs])
        color = model_color(label)
        ax.scatter(x, y, color=color, alpha=0.3, s=6, rasterized=True)
        if len(x) >= 2 and np.ptp(x) > 0:
            coeffs = np.polyfit(x, y, 1)
            x_line = np.linspace(x.min(), x.max(), 200)
            ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1.5,
                    linestyle="--", label=f"slope={coeffs[0]:.3f}")
            ax.legend(fontsize=6, loc="upper left")
        if row == 0:
            ax.set_title(label, fontsize=9)
        if col == 0:
            ax.set_ylabel(f"Absolute error\n({hd_name})", fontsize=8)
        if row == 1:
            ax.set_xlabel(hd_name, fontsize=8)

plt.suptitle("All models' test error vs CIN / CINpp Hasse graph diameter", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()